# Query Reformulation and Multi-Hop Retrieval

Reformulate, hop, and route: agentic retrieval beyond a single similarity search, on LangChain + LangGraph

The corpus deliberately spans **two domains** (a small company-ops wiki and a small math/numeric facts table) so a router has something real to route between, and so HyDE, decomposition, and tool-augmented retrieval each have a scenario where they visibly help.

## Setup

Let's wire the model-agnostic stack and build a deliberately two-domain corpus.

Now, we will
- assemble a 12-document corpus that spans **company ops** facts AND **math / numeric** facts so the router has two distinct domains to choose between,
- index it in an `InMemoryVectorStore` and expose a `retrieve(query, k)` helper,
- define three evaluation questions that each stress a different reformulation / routing decision.

In [1]:
# !pip install -q langchain langchain-google-genai langchain-openai langchain-anthropic langgraph langchain-community

from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.tools import tool
from typing import TypedDict, List, Optional, Literal
from pydantic import BaseModel, Field
import os, json
# Keep the API keys for the models of choice in the loaded env file
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# os.environ['GEMINI_API_KEY']  # the variable for API key

llm   = init_chat_model('gemini-2.5-flash-lite', model_provider='google_genai', temperature=0)
embed = init_embeddings('sentence-transformers/all-MiniLM-L6-v2', provider='huggingface')

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Now the corpus. Twelve chunks split across two domains. Documents *d1..d7* are **company ops** facts engineered as a multi-hop chain (`Atlas → Helios → Snowflake → 400-credit quota`). Documents *m1..m5* are short **math / numeric** facts (constants and conversion rates) that the router should send to a calculator tool, not to the wiki retriever.

In [3]:
# 12-doc corpus deliberately split across two domains.
# d1..d7  -> company-ops chain (Atlas -> Helios -> Snowflake -> 400-credit quota)
# m1..m5  -> math / numeric facts (calculator-domain, not wiki-domain)
CORPUS = [
    {'id': 'd1', 'domain': 'docs', 'text': 'Project Atlas is the flagship customer-analytics product, owned by the Data Platform group.'},
    {'id': 'd2', 'domain': 'docs', 'text': 'Atlas depends on the Helios ingestion service for all upstream event data.'},
    {'id': 'd3', 'domain': 'docs', 'text': 'Helios streams data into the Snowflake warehouse via a Kafka -> Snowpipe bridge.'},
    {'id': 'd4', 'domain': 'docs', 'text': 'The Snowflake warehouse enforces a per-customer compute quota of 400 credits per month for any landing pipeline.'},
    {'id': 'd5', 'domain': 'docs', 'text': 'Priya Raman leads the Atlas product team and reports to Arjun Mehta, Director of Data Platform.'},
    {'id': 'd6', 'domain': 'docs', 'text': 'Karthik Iyer leads Ingestion Platform and owns Helios; he reports to Arjun Mehta.'},
    {'id': 'd7', 'domain': 'docs', 'text': 'Nadia Haq leads the Warehouse team and enforces the Snowflake compute quota and cost-allocation tags.'},
    {'id': 'm1', 'domain': 'math', 'text': 'One Snowflake credit currently costs 3 US dollars on the standard tier.'},
    {'id': 'm2', 'domain': 'math', 'text': 'There are 1024 megabytes in one gigabyte and 1024 gigabytes in one terabyte.'},
    {'id': 'm3', 'domain': 'math', 'text': 'A typical Helios partition processes 250 events per second on a standard worker.'},
    {'id': 'm4', 'domain': 'math', 'text': 'The Atlas SLA is 99.9 percent monthly uptime, which allows roughly 43.2 minutes of downtime per month.'},
    {'id': 'm5', 'domain': 'math', 'text': 'Pi is approximately 3.14159 and e is approximately 2.71828.'},
]

docs = [Document(page_content=c['text'], metadata={'id': c['id'], 'domain': c['domain']}) for c in CORPUS]
store = InMemoryVectorStore.from_documents(docs, embedding=embed)
print('Corpus size:', len(CORPUS))
print('Domains    :', sorted({c['domain'] for c in CORPUS}))

Corpus size: 12
Domains    : ['docs', 'math']


Now the retrieval primitive. `retrieve(query, k)` is the **same function** every later section calls, so any accuracy delta we see is attributable to *how the query was reformulated* or *which tool the router chose*, not to a smarter retriever.

In [4]:
def retrieve(query: str, k: int = 3) -> List[Document]:
    """Top-k similarity retrieval shared by every later section. The variable that changes
    across sections is WHAT we feed in (raw question vs HyDE answer vs sub-question), NOT this function."""
    return store.similarity_search(query, k=k)

# Three evaluation questions, each chosen to expose a different failure mode:
Q_RAW    = 'Who runs Atlas?'                                                          # short, lexically thin -> HyDE will help
Q_HOP    = 'What monthly compute quota applies to the warehouse used by the upstream service of Atlas?'  # multi-hop -> decomposition
Q_NUM    = 'If Helios writes 400 credits worth of data this month, how many US dollars is that?'         # numeric -> calculator tool
print('Q_RAW :', Q_RAW)
print('Q_HOP :', Q_HOP)
print('Q_NUM :', Q_NUM)

Q_RAW : Who runs Atlas?
Q_HOP : What monthly compute quota applies to the warehouse used by the upstream service of Atlas?
Q_NUM : If Helios writes 400 credits worth of data this month, how many US dollars is that?


*Tip: as an exercise, replace `InMemoryVectorStore` with a proper vector DB like `Chroma` or `FAISS`. Every section below works unchanged because the LangChain `VectorStore` interface is the same.*

## HyDE: retrieving against a hypothetical answer

Let's start with the simplest reformulation: **HyDE** (Hypothetical Document Embeddings). The user's raw question is often a bad embedding query because questions and answers live in different lexical neighborhoods. HyDE asks the LLM to draft a hypothetical answer first, then embeds *that* and retrieves with it.

Now, we will
- run a baseline retrieval on the raw `Q_RAW` and inspect what comes back,
- prompt the LLM to draft a short hypothetical answer to the same question (no retrieval, just LLM imagination),
- retrieve against the **embedding of the hypothetical answer** and compare the document ids,
- confirm that HyDE pulls in the correct chunk (`d5`, which has 'Priya Raman leads Atlas') even when the raw query misses it.

In [5]:
# Baseline: retrieve against the raw question.
raw_hits = retrieve(Q_RAW, k=3)
print('---- raw question top-3 ----')
for d in raw_hits:
    print(f"  [{d.metadata['id']}] {d.page_content[:90]}...")

---- raw question top-3 ----
  [d5] Priya Raman leads the Atlas product team and reports to Arjun Mehta, Director of Data Plat...
  [d2] Atlas depends on the Helios ingestion service for all upstream event data....
  [d1] Project Atlas is the flagship customer-analytics product, owned by the Data Platform group...


Now the HyDE prompt. Notice it tells the LLM to invent a plausible-sounding answer **even if it is wrong**. We are not using the answer for anything except as a richer embedding query.

In [6]:
HYDE_PROMPT = """You are drafting a HYPOTHETICAL one-paragraph answer to a user question.

It is fine if you do not know the true answer — write the kind of paragraph you WOULD expect
to find in a wiki or runbook that addresses this question. Use full sentences, named entities,
and concrete nouns. Do not hedge.

Question: {q}
Hypothetical answer:"""

def hyde_query(question: str) -> str:
    """Ask the LLM for a hypothetical answer. We will embed THIS, not the question.
    The hypothetical answer lives in the same lexical neighborhood as real answers,
    which is why retrieving against its embedding tends to outperform retrieving against the raw question."""
    return llm.invoke(HYDE_PROMPT.format(q=question)).content

Let's draft a hypothetical answer for `Q_RAW` and look at it. We expect a confident-sounding paragraph that may even fabricate names, that is fine. The point is that its **embedding** lands closer to real answer chunks.

In [7]:
hypo = hyde_query(Q_RAW)
print('---- hypothetical answer (truncated) ----')
print(hypo[:240] + '...')

hyde_hits = retrieve(hypo, k=3)
print('\n---- HyDE top-3 (retrieved against hypothetical embedding) ----')
for d in hyde_hits:
    print(f"  [{d.metadata['id']}] {d.page_content[:90]}...")

print('\n=== ID DELTA ===')
print('raw  ids:', [d.metadata['id'] for d in raw_hits])
print('HyDE ids:', [d.metadata['id'] for d in hyde_hits])

---- hypothetical answer (truncated) ----
The Atlas project is managed by the Atlas Foundation, a non-profit organization dedicated to fostering open-source development and community collaboration. The Foundation's board of directors, comprised of elected representatives from the A...

---- HyDE top-3 (retrieved against hypothetical embedding) ----
  [d1] Project Atlas is the flagship customer-analytics product, owned by the Data Platform group...
  [d5] Priya Raman leads the Atlas product team and reports to Arjun Mehta, Director of Data Plat...
  [d2] Atlas depends on the Helios ingestion service for all upstream event data....

=== ID DELTA ===
raw  ids: ['d5', 'd2', 'd1']
HyDE ids: ['d1', 'd5', 'd2']


*Tip: HyDE is cheap (one extra LLM call) and never needs you to re-embed the corpus. Keep `temperature=0` on the HyDE call, diversity should come from the technique, not from sampling noise.*

## Decomposition: `SubQueries` from a multi-hop question

Now let's tackle the harder reformulation: **decomposition**. `Q_HOP` is a single sentence that hides three lookups: *upstream service of Atlas?*, *which warehouse?*, *what is the quota?*. We force the LLM to split it into sub-questions via `with_structured_output`, then retrieve evidence for each.

Now, we will
- define a Pydantic `SubQueries` schema so the LLM's decomposition is machine-readable,
- bind it via `llm.with_structured_output(SubQueries)` and decompose `Q_HOP` into 2-3 sub-questions,
- print the sub-questions to confirm they cover the hop chain,
- and (in the next section) feed them into a multi-hop loop that accumulates evidence.

In [8]:
class SubQueries(BaseModel):
    """Decomposition schema. The LLM must return a small list of focused sub-questions
    that, taken together, cover the original multi-hop question."""
    queries: List[str] = Field(
        description='Two or three focused sub-questions that decompose the original question, in retrieval order.',
        min_length=2, max_length=3,
    )

The schema **is** the contract. `min_length=2, max_length=3` keeps the loop bounded. We will not accidentally let the LLM emit nine sub-questions and run up the bill.

In [9]:
DECOMPOSE_PROMPT = """You are a retrieval planner. The user question below requires looking up
several facts in a wiki and stitching them together. Decompose it into 2-3 focused sub-questions
such that each sub-question can be answered by a single short paragraph. Order the sub-questions
from the first hop (which entity to resolve first) to the last hop (the actual user-facing answer).

Question: {q}
"""

decomposer = llm.with_structured_output(SubQueries)

def decompose(question: str) -> SubQueries:
    """Run the structured-output decomposition. The result is a typed object,
    so downstream code never has to regex-parse free-text bullets."""
    return decomposer.invoke(DECOMPOSE_PROMPT.format(q=question))

subs = decompose(Q_HOP)
print('=== SUB-QUERIES ===')
for i, q in enumerate(subs.queries, 1):
    print(f'  {i}. {q}')

=== SUB-QUERIES ===
  1. What is the upstream service of Atlas?
  2. What warehouse does the upstream service of Atlas use?
  3. What is the monthly compute quota for that warehouse?


## Iterative multi-hop retrieval loop

Now let's use those sub-questions as the **hop sequence**. We iterate over them, retrieve top-k for each, and dedupe by document id so we never waste a slot on a chunk we already have. A plain `for` loop is enough. We do not need a graph for this shape because the hop count is fixed by the decomposition.

Now, we will
- define `multi_hop_retrieve(question)` that decomposes the question, then loops over sub-queries,
- accumulate evidence across iterations, deduping by `metadata.id`,
- print a per-iteration trace under `hop N` banners so the hop sequence is visible,
- generate a final answer from the merged evidence and confirm the **400-credit** number is recovered.

In [10]:
def format_context(docs: List[Document]) -> str:
    """Serialise retrieved docs into a numbered context block. Kept deterministic so
    different reformulation strategies are directly comparable downstream."""
    return '\n'.join(f"[{d.metadata['id']}] {d.page_content}" for d in docs)

Now the loop itself. The pattern is `decompose -> for sub in subs: retrieve + dedupe -> synthesise`. The dedupe step is what stops the same first-hop chunk from re-winning every iteration.

In [11]:
ANSWER_PROMPT = """Answer the user's QUESTION using ONLY the EVIDENCE below. Cite [doc_ids] inline.

QUESTION: {q}

EVIDENCE:
{ctx}
"""

def multi_hop_retrieve(question: str, k: int = 3) -> dict:
    """Iterative multi-hop retrieval. Decompose into sub-questions, retrieve for each,
    dedupe by id across iterations, then synthesise a final answer over merged evidence.
    The hop count is BOUNDED by the SubQueries schema (max 3) — no runaway loops."""
    subs = decompose(question)
    evidence: List[Document] = []
    seen_ids: set = set()
    trace: List[dict] = []

    for hop, sub in enumerate(subs.queries, 1):
        hits = retrieve(sub, k=k)
        new = [d for d in hits if d.metadata['id'] not in seen_ids]
        seen_ids.update(d.metadata['id'] for d in new)
        evidence.extend(new)
        trace.append({'hop': hop, 'sub_query': sub, 'new_ids': [d.metadata['id'] for d in new]})

    answer = llm.invoke(ANSWER_PROMPT.format(q=question, ctx=format_context(evidence))).content
    return {'sub_queries': subs.queries, 'evidence_ids': sorted(seen_ids), 'answer': answer, 'trace': trace}

Time to run it on `Q_HOP` and read the trace. We are expecting two or three hops, each surfacing a NEW chunk, and a final answer that contains the **400 credits** number — which the raw-question retrieval would never have surfaced.

In [12]:
result = multi_hop_retrieve(Q_HOP)

print('=== SUB-QUERIES ===')
for i, q in enumerate(result['sub_queries'], 1):
    print(f'  {i}. {q}')

for step in result['trace']:
    print(f"\n---- hop {step['hop']} ----")
    print(f"  sub_query: {step['sub_query']}")
    print(f"  new ids  : {step['new_ids']}")

print('\n=== EVIDENCE IDS ===')
print(result['evidence_ids'])

print('\n=== FINAL ANSWER ===')
print(result['answer'])

=== SUB-QUERIES ===
  1. What is the upstream service of Atlas?
  2. What warehouse does the upstream service of Atlas use?
  3. What is the monthly compute quota for that warehouse?

---- hop 1 ----
  sub_query: What is the upstream service of Atlas?
  new ids  : ['d2', 'd1', 'm4']

---- hop 2 ----
  sub_query: What warehouse does the upstream service of Atlas use?
  new ids  : ['d5']

---- hop 3 ----
  sub_query: What is the monthly compute quota for that warehouse?
  new ids  : ['d4', 'd7']

=== EVIDENCE IDS ===
['d1', 'd2', 'd4', 'd5', 'd7', 'm4']

=== FINAL ANSWER ===
The Snowflake warehouse enforces a per-customer compute quota of 400 credits per month for any landing pipeline [d4].


*Tip: decomposition shines on questions whose sub-parts each have a clear lexical anchor. If the sub-questions still feel ambiguous, you can stack HyDE on top; embed the hypothetical answer of each sub-question, not the sub-question itself.*

## Tool-augmented retrieval with `bind_tools`

Now let's give the LLM a choice. We expose **two retrievers as tools** (one for the `docs` domain, one for the `math` domain) plus a tiny **calculator** tool, then bind them with `llm.bind_tools([...])` and let the model pick which to call.

Now, we will
- define three `@tool`-decorated functions: `search_docs`, `search_math`, and `calculator`,
- bind them to the LLM with `llm.bind_tools([...])` so the model can emit `tool_calls`,
- invoke the bound LLM on `Q_NUM` (which needs both a fact lookup AND arithmetic),
- inspect `response.tool_calls` to see which tools the LLM chose and with what arguments.

In [13]:
@tool
def search_docs(query: str) -> str:
    """Search the company-ops wiki (Atlas, Helios, Snowflake, org chart, quotas).
    Use this for narrative questions about people, services, ownership, and policies."""
    hits = [d for d in retrieve(query, k=4) if d.metadata['domain'] == 'docs'][:3]
    return format_context(hits) if hits else 'No matching docs.'

@tool
def search_math(query: str) -> str:
    """Search the math / numeric facts table (constants, conversion rates, unit prices).
    Use this when you need to look up a numeric fact before computing with it."""
    hits = [d for d in retrieve(query, k=4) if d.metadata['domain'] == 'math'][:3]
    return format_context(hits) if hits else 'No matching math facts.'

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '400 * 3' or '(1024 ** 2)'.
    Use this AFTER you have looked up the numeric facts you need."""
    # Tiny safe evaluator: digits, operators, parens, decimal points and spaces only.
    allowed = set('0123456789+-*/()., ')
    expr = ''.join(c for c in expression if c in allowed)
    return str(eval(expr)) if expr else 'empty expression'

Three tools, three docstrings. The docstring is the tool description the LLM sees, so the **tool selection quality depends directly on how clearly we tell it WHEN to use each tool**. Notice we say 'use this AFTER you have looked up the numeric facts' on `calculator`.

In [14]:
tools = [search_docs, search_math, calculator]
tool_llm = llm.bind_tools(tools)

print('Tools bound:', [t.name for t in tools])

Tools bound: ['search_docs', 'search_math', 'calculator']


Now invoke the bound LLM on `Q_NUM` and inspect its `tool_calls`. We expect the model to call `search_math` (to look up the credit-to-dollar rate from `m1`) and `calculator` (to multiply `400 * 3`) — possibly in parallel.

In [15]:
response = tool_llm.invoke(Q_NUM)

print('=== RESPONSE CONTENT ===')
print((response.content or '(no content; expecting tool_calls)')[:240])

print('\n=== TOOL CALLS ===')
for tc in response.tool_calls:
    print(f"  -> {tc['name']}({tc['args']})")

print('\n=== EXECUTING THE TOOL CALLS ===')
tool_by_name = {t.name: t for t in tools}
for tc in response.tool_calls:
    out = tool_by_name[tc['name']].invoke(tc['args'])
    print(f"---- {tc['name']} ----")
    print(str(out)[:200])

=== RESPONSE CONTENT ===
(no content; expecting tool_calls)

=== TOOL CALLS ===
  -> search_math({'query': 'credits to USD conversion rate'})

=== EXECUTING THE TOOL CALLS ===
---- search_math ----
[m1] One Snowflake credit currently costs 3 US dollars on the standard tier.
[m4] The Atlas SLA is 99.9 percent monthly uptime, which allows roughly 43.2 minutes of downtime per month.


*Tip: `bind_tools` only **declares** the tools; the LLM emits `tool_calls` but does not execute them. You (or a `ToolNode` in LangGraph) execute the calls and feed the results back. The split gives you a chance to validate / log / rate-limit before any side effect runs.*

## Routing with `RouteDecision`

Tool-binding lets the LLM choose freely. Sometimes we want a **smaller, cheaper, deterministic** decision instead. A router that classifies the question into one of a fixed set of destinations. Let's wire one with `with_structured_output`.

Now, we will
- define a `RouteDecision` Pydantic schema with `route: Literal['docs', 'calc', 'general']` and a one-line `reason`,
- bind it via `llm.with_structured_output(RouteDecision)` so routing is a single typed call,
- run three example queries (one for each route) through the router and dispatch to the matching backend,
- print the route, the reason, and a short answer for each.

In [16]:
class RouteDecision(BaseModel):
    """Where should this question be answered? One of three destinations.
    The Literal[...] type keeps branching stable across runs — the LLM cannot
    invent a fourth route and crash the dispatch."""
    route: Literal['docs', 'calc', 'general'] = Field(
        description="'docs' for company-ops wiki questions, 'calc' for numeric / arithmetic questions, 'general' otherwise."
    )
    reason: str = Field(description='One short sentence justifying the route.')

The schema doubles as documentation for the LLM. We add a `reason` field so every routing decision is auditable. In production you log this alongside the question to build the offline dataset for tuning the router prompt.

In [17]:
ROUTER_PROMPT = """You are a router. Read the QUESTION and choose ONE route:

- 'docs'    : the answer is in the company-ops wiki (people, services, ownership, quotas).
- 'calc'    : the question requires arithmetic on numeric facts (multiplications, conversions).
- 'general' : neither of the above — answer from your own knowledge.

QUESTION: {q}"""

router = llm.with_structured_output(RouteDecision)

def route_query(question: str) -> RouteDecision:
    """Classify the question with structured output. One LLM call, typed result."""
    return router.invoke(ROUTER_PROMPT.format(q=question))

def dispatch(question: str) -> dict:
    """Route the question, then call the matching backend. The router does NOT execute the answer —
    it only picks the destination, which is what makes it cheap and inspectable."""
    decision = route_query(question)
    if decision.route == 'docs':
        answer = search_docs.invoke({'query': question})
    elif decision.route == 'calc':
        answer = tool_llm.invoke(question).content or '(see tool_calls)'
    else:
        answer = llm.invoke(question).content
    return {'question': question, 'route': decision.route, 'reason': decision.reason, 'answer': answer}

Now three example queries, one per route. We expect `Q_HOP` -> `docs`, `Q_NUM` -> `calc`, and a generic question to fall through to `general`.

In [18]:
examples = [
    Q_HOP,
    Q_NUM,
    'In one sentence, what is retrieval-augmented generation?',
]

for q in examples:
    out = dispatch(q)
    print(f"\n---- route: {out['route']} ----")
    print('Q     :', out['question'])
    print('reason:', out['reason'])
    print('answer:', (out['answer'] or '')[:200].replace('\n', ' ') + '...')


---- route: docs ----
Q     : What monthly compute quota applies to the warehouse used by the upstream service of Atlas?
reason: The question asks about a specific compute quota for a service, which is likely documented in the company-ops wiki.
answer: [d4] The Snowflake warehouse enforces a per-customer compute quota of 400 credits per month for any landing pipeline. [d2] Atlas depends on the Helios ingestion service for all upstream event data. [d...

---- route: calc ----
Q     : If Helios writes 400 credits worth of data this month, how many US dollars is that?
reason: The question requires a conversion from credits to US dollars, which involves a numerical calculation.
answer: (see tool_calls)...

---- route: general ----
Q     : In one sentence, what is retrieval-augmented generation?
reason: The question asks for a definition of a technical term, which can be answered from general knowledge.
answer: Retrieval-augmented generation (RAG) is a technique that enhances large languag

*Tip: keep the router on `temperature=0` and version the router prompt like code. Every prompt change is a model change. Diff and tag them so you can trace a regression back to its commit.*

## Tips and common pitfalls

### Adopting reformulation and routing in production
- Run reformulation strategies in **parallel**, not in series. HyDE, decomposition, and a raw-query baseline can all retrieve concurrently; merge by id with a stable rank fusion (RRF) at the end.
- Cap the number of sub-questions in your decomposition schema. `max_length=3` in `SubQueries` is a teaching value, pick a number that bounds your worst-case cost per user query.
- Always log the **router decision + reason + final tool calls** for every request. That log is the offline-eval dataset you will use to retune the router prompt; without it you are flying blind.
- Keep `temperature=0` on every reformulation, decomposition, and routing call. Diversity should come from the schema, not from sampling noise.

### Common pitfalls in tool-augmented retrieval
- The LLM will sometimes call a tool with badly-typed arguments. The `@tool` decorator validates against the function's type hints, but you still want a top-level guard around your dispatch loop. <br>Mitigation: write explicit Pydantic argument schemas for any tool whose signature is non-trivial.
- `bind_tools` only **declares** the tools. The LLM emits `tool_calls` but you execute them. Forgetting to feed the tool results back as a `ToolMessage` is the #1 reason agents 'silently give up' after one call.
- Tool docstrings ARE the tool descriptions the LLM sees. A vague docstring produces vague routing; rewrite docstrings as if the LLM were a junior engineer reading them for the first time.
- Watch for **route collapse**. The LLM picking the same route for every question regardless of content. Mitigation: hold out a small balanced set of questions per route and re-eval the router after every prompt change.